In [ ]:
# STEP 0: Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    project_path = '/content/gdrive/MyDrive/feral-cat-fcm'
except ImportError:
    project_path = os.path.abspath('feral-cat-fcm')  # running locally

# Define dataset and output paths
dataset_path = f'{project_path}/data/yolo_25-15-60'
output_path = f'{project_path}/outputs/yolo_tuning'

In [ ]:
# STEP 1: Install YOLOv12 via Ultralytics
!pip install -U ultralytics

In [ ]:
# STEP 2: Train m, l and x at 896 px for 150 epochs, then score on val and test
import pandas as pd
from ultralytics import YOLO

scores = []
for weights in ['yolo12m.pt', 'yolo12l.pt', 'yolo12x.pt']:
    for batch in [8, 16]:
        name = f"{weights.replace('.pt', '')}_batch{batch}"
        model = YOLO(weights)
        model.train(
            data=f'{dataset_path}/data.yaml',
            epochs=150,
            imgsz=896,
            batch=batch,
            project=output_path,
            name=name,
            exist_ok=True
        )
        model = YOLO(f'{output_path}/{name}/weights/best.pt')
        for split, conf in [('val', 0.001), ('test', 0.15), ('test', 0.25), ('test', 0.35), ('test', 0.40)]:
            m = model.val(data=f'{dataset_path}/data.yaml', split=split, conf=conf, imgsz=896, batch=batch,
                          project=output_path, name=f'{name}_{split}_conf{conf}', exist_ok=True)
            scores.append([name, split, conf, m.box.mp, m.box.mr, m.box.map50, m.box.map])

In [ ]:
# STEP 3: Save scores
scores = pd.DataFrame(scores, columns=['model', 'split', 'conf', 'precision', 'recall', 'mAP50', 'mAP50-95']).round(3)
scores.to_csv(f'{output_path}/scores.csv', index=False)
scores